# Train Forecasting Model

## Imports

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

## Loading Data

In [2]:
PROCESSED_DATA_DIR= '../data/processed'
MODELS_DIR= '../models'

In [3]:
data_path= os.path.join(PROCESSED_DATA_DIR, 'forecasting_training_data.csv')

In [4]:
df= pd.read_csv(filepath_or_buffer= data_path)

In [5]:
df.head()

,sale_date,category,units_sold,daily_revenue
0,2016-09-15,health_beauty,3,134.97
1,2016-10-03,fashion_shoes,1,29.99
2,2016-10-03,furniture_decor,2,194.80
3,2016-10-03,sports_leisure,2,58.39
4,2016-10-03,toys,1,128.90


In [6]:
df.describe()

,units_sold,daily_revenue
count,18311.000000,18311.000000
mean,5.932936,712.442297
std,7.285815,969.197817
min,1.000000,3.850000
25%,1.000000,119.730000
50%,3.000000,349.900000
75%,8.000000,923.335000
max,192.000000,17667.020000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_date      18311 non-null  object 
 1   category       18311 non-null  object 
 2   units_sold     18311 non-null  int64  
 3   daily_revenue  18311 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 572.3+ KB


In [8]:
df.shape

(18311, 4)

## Feature Engineering

In [9]:
# Converting sale_date to DateTime:
df['sale_date'] = pd.to_datetime(df['sale_date'])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   sale_date      18311 non-null  datetime64[ns]
 1   category       18311 non-null  object        
 2   units_sold     18311 non-null  int64         
 3   daily_revenue  18311 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 572.3+ KB


In [11]:
# Sorting DataFrame by category and sale_date:
df= df.sort_values(['category', 'sale_date']).reset_index(drop= True)

In [12]:
df.head(10)

,sale_date,category,units_sold,daily_revenue
0,2017-01-23,agro_industry_and_commerce,2,43.98
1,2017-01-31,agro_industry_and_commerce,1,21.99
2,2017-02-05,agro_industry_and_commerce,1,21.99
3,2017-02-08,agro_industry_and_commerce,1,21.99
4,2017-02-12,agro_industry_and_commerce,1,92.90
5,2017-02-13,agro_industry_and_commerce,1,21.99
6,2017-02-16,agro_industry_and_commerce,1,21.99
7,2017-02-18,agro_industry_and_commerce,1,21.99
8,2017-02-21,agro_industry_and_commerce,1,21.99
9,2017-03-17,agro_industry_and_commerce,1,59.99


In [18]:
# Extracting Standard Time-Series Features:
df['year'] = df['sale_date'].dt.year
df['month'] = df['sale_date'].dt.month
df['day_of_week'] = df['sale_date'].dt.dayofweek
df['day_of_year'] = df['sale_date'].dt.dayofyear

In [19]:
df.head()

,sale_date,category,units_sold,daily_revenue,year,month,day_of_week,day_of_year
0,2017-01-23,agro_industry_and_commerce,2,43.98,2017,1,0,23
1,2017-01-31,agro_industry_and_commerce,1,21.99,2017,1,1,31
2,2017-02-05,agro_industry_and_commerce,1,21.99,2017,2,6,36
3,2017-02-08,agro_industry_and_commerce,1,21.99,2017,2,2,39
4,2017-02-12,agro_industry_and_commerce,1,92.90,2017,2,6,43


In [20]:
# Adding Lags and EWMA:

# 1. Yesterday:
df['lag_1_revenue'] = df.groupby('category')['daily_revenue'].shift(1)

# 2. Same Day, Last Week:
df['lag_7_revenue'] = df.groupby('category')['daily_revenue'].shift(7)

# 3. EWMA prioritizing last 7 days:
df['ewma_7_revenue'] = df.groupby('category')['daily_revenue'].transform(
    lambda x: x.ewm(span= 7, adjust=False).mean().shift(1)
)

In [21]:
df.head()

,sale_date,category,units_sold,daily_revenue,year,month,day_of_week,day_of_year,lag_1_revenue,lag_7_revenue,ewma_7_revenue
0,2017-01-23,agro_industry_and_commerce,2,43.98,2017,1,0,23,NaN,NaN,NaN
1,2017-01-31,agro_industry_and_commerce,1,21.99,2017,1,1,31,43.98,NaN,43.980000
2,2017-02-05,agro_industry_and_commerce,1,21.99,2017,2,6,36,21.99,NaN,38.482500
3,2017-02-08,agro_industry_and_commerce,1,21.99,2017,2,2,39,21.99,NaN,34.359375
4,2017-02-12,agro_industry_and_commerce,1,92.90,2017,2,6,43,21.99,NaN,31.267031


In [23]:
# Dropping Null values Created by Lag Features:
df= df.dropna().reset_index(drop= True)

In [24]:
df.head(10)

,sale_date,category,units_sold,daily_revenue,year,month,day_of_week,day_of_year,lag_1_revenue,lag_7_revenue,ewma_7_revenue
0,2017-02-18,agro_industry_and_commerce,1,21.99,2017,2,5,49,21.99,43.98,35.875466
1,2017-02-21,agro_industry_and_commerce,1,21.99,2017,2,1,52,21.99,21.99,32.404100
2,2017-03-17,agro_industry_and_commerce,1,59.99,2017,3,4,76,21.99,21.99,29.800575
3,2017-03-20,agro_industry_and_commerce,1,22.00,2017,3,0,79,59.99,21.99,37.347931
4,2017-05-06,agro_industry_and_commerce,1,59.99,2017,5,5,126,22.00,92.90,33.510948
5,2017-05-08,agro_industry_and_commerce,1,589.99,2017,5,0,128,59.99,21.99,40.130711
6,2017-05-20,agro_industry_and_commerce,1,869.97,2017,5,5,140,589.99,21.99,177.595533
7,2017-05-28,agro_industry_and_commerce,1,59.99,2017,5,6,148,869.97,21.99,350.689150
8,2017-06-26,agro_industry_and_commerce,1,1390.00,2017,6,0,177,59.99,21.99,278.014363
9,2017-07-14,agro_industry_and_commerce,1,1180.00,2017,7,4,195,1390.00,59.99,556.010772
